<a href="https://www.kaggle.com/code/emmanuelniyioriolowo/data-ingestion?scriptVersionId=281979582" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ncdc-table/ncdc_table.html


In [2]:
!pip install reportlab
!pip install fpdf
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 21.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=4788fb096e8ee858b5eacb1fd99646628fd1eda6d736faee339e4c7ab6856ee1
  Stored in directory: /root/.cache/pip/wheels/65/4f/66/bbda9866da446a72e206d6484cd97381cbc7859a7068541c36
Successfully built fpdf
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.7 MB/s eta 0:00:00


In [3]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import date
from reportlab.pdfgen import canvas
import re
from datetime import datetime, timedelta
from fpdf import FPDF  

In [4]:
BASE_URL = "https://ncdc.gov.ng"
HTML_FILE = "/kaggle/input/ncdc-table/ncdc_table.html"
SAVE_DIR = "/kaggle/working/ncdc_reports"
START_SN = 305  # Starting SN (week 1 of 2020)
START_YEAR = 2020
START_WEEK = 1

os.makedirs(SAVE_DIR, exist_ok=True)

In [5]:
# Load saved HTML table
with open(HTML_FILE, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

# Find all table rows 
rows = soup.find_all("tr")[1:]  # skip the first row (header)

# Store SN and PDF link
pdf_data = []

for row in rows:
    cols = row.find_all("td")
    if len(cols) >= 3:
        sn = cols[0].get_text(strip=True)
        title = cols[1].get_text(strip=True)
        link_tag = cols[2].find("a", href=True)
        if link_tag:
            pdf_url = urljoin(BASE_URL, link_tag['href'])
            pdf_data.append((int(sn), title, pdf_url))

# Sort by SN (ascending)
pdf_data.sort(key=lambda x: x[0])

print(f"Total PDFs found: {len(pdf_data)}")

# Print results
for sn, title, url in pdf_data[:9]:
    print(f"SN {sn} | Title: {title} | URL: {url}")

Total PDFs found: 441
SN 1 | Title: An update of Lassa fever outbreak in Nigeria for Week 45 | URL: https://ncdc.gov.ng/themes/common/files/sitreps/471dd6384cd4f288167c0881efee40ee.pdf
SN 2 | Title: An update of Lassa fever outbreak in Nigeria for Week 44 | URL: https://ncdc.gov.ng/themes/common/files/sitreps/9910b5860f5b9c377cf14a8992d6905b.pdf
SN 3 | Title: An update of Lassa fever outbreak in Nigeria for Week 43 | URL: https://ncdc.gov.ng/themes/common/files/sitreps/55b33161bbfe39fd72039808875e5c45.pdf
SN 4 | Title: An update of Lassa fever outbreak in Nigeria for Week 42 | URL: https://ncdc.gov.ng/themes/common/files/sitreps/91f559d691c7cd411f7058f1496eef6f.pdf
SN 5 | Title: An update of Lassa fever outbreak in Nigeria for Week 41 | URL: https://ncdc.gov.ng/themes/common/files/sitreps/68f0ee9000f3084b74ed892be0d3f79f.pdf
SN 6 | Title: An update of Lassa fever outbreak in Nigeria for Week 40 | URL: https://ncdc.gov.ng/themes/common/files/sitreps/ded728ef465f33739ad7928d327a2929.pdf


In [6]:
# Helper functions 
def create_placeholder_pdf(filename, week_info):
    """Create a placeholder PDF for missing weeks"""
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    pdf.cell(200, 10, txt=f"Report Not Available", ln=1, align='C')
    pdf.cell(200, 10, txt=f"Week {week_info}", ln=1, align='C')
    pdf.cell(200, 10, txt=f"Placeholder for missing report", ln=1, align='C')
    pdf.output(os.path.join(SAVE_DIR, filename))

def get_week_year_from_sn(sn):
    """Calculate year and week from SN number"""
    weeks_passed = sn - START_SN
    current_date = datetime(START_YEAR, 1, 1)
    
    # Find the first Thursday of the year (ISO week start)
    while current_date.weekday() != 3:  # 3 = Thursday
        current_date += timedelta(days=1)
    
    # Add the weeks passed
    target_date = current_date + timedelta(weeks=weeks_passed)
    year = target_date.isocalendar()[0]
    week = target_date.isocalendar()[1]
    
    return year, week

def download_pdf(url, filename):
    """Download actual PDF from URL"""
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        with open(os.path.join(SAVE_DIR, filename), 'wb') as f:
            f.write(response.content)
        print(f"✓ Downloaded: {filename}")
        return True
    except Exception as e:
        print(f"✗ Failed to download {filename}: {e}")
        return False


In [7]:
# Create a mapping of (year, week) to PDF data for easy lookup
available_reports = {}
for sn, title, url in pdf_data:
    if sn >= START_SN:
        year, week = get_week_year_from_sn(sn)
        available_reports[(year, week)] = (sn, title, url)
        print(f"SN {sn} → {year}-W{week:02d}: {title}")

# Find the range of weeks to process
min_sn = START_SN
max_sn = max(sn for sn, _, _ in pdf_data if sn >= START_SN)

print(f"\nProcessing SN range: {min_sn} to {max_sn}")

# Process all weeks from START_SN downward to min_sn
# for sn in range(START_SN, min_sn - 1, -1):
for sn in range(START_SN, START_SN - 10, -1):
    # Calculate the week offset from START_SN
    week_offset = START_SN - sn  # This will be 0 for SN 305, 1 for SN 304, etc.
    year = START_YEAR
    week = START_WEEK + week_offset
    
    # Handle year transitions (52 weeks per year)
    while week > 52:
        year += 1
        week -= 52
    
    filename = f"{year}-W{week:02d}.pdf"
    
    # Find the PDF data for this SN
    pdf_info = next((item for item in pdf_data if item[0] == sn), None)
    
    if pdf_info:
        # Download actual PDF
        _, title, url = pdf_info
        print(f"SN {sn}: Downloading {filename} - {title}")
        download_pdf(url, filename)
    else:
        # Create placeholder for missing week
        print(f"SN {sn}: Creating placeholder for {filename} (missing)")
        create_placeholder_pdf(filename, f"{year} Week {week}")

print(f"\nProcessing complete!")
print(f"Files saved to: {SAVE_DIR}")

# Print summary
downloaded_count = len([f for f in os.listdir(SAVE_DIR) if f.endswith('.pdf')])
print(f"Total PDF files in directory: {downloaded_count}")

SN 305 → 2020-W01: An update of Lassa fever outbreak in Nigeria for Week 1
SN 306 → 2020-W02: An update of Lassa fever outbreak in Nigeria for Week 52
SN 307 → 2020-W03: An update of Lassa fever outbreak in Nigeria for Week 51
SN 308 → 2020-W04: An update of Lassa fever outbreak in Nigeria for Week 50
SN 309 → 2020-W05: An update of Lassa fever outbreak in Nigeria for Week 49
SN 310 → 2020-W06: An update of Lassa fever outbreak in Nigeria for Week 48
SN 311 → 2020-W07: An update of Lassa fever outbreak in Nigeria for Week 47
SN 312 → 2020-W08: An update of Lassa fever outbreak in Nigeria for Week 46
SN 313 → 2020-W09: An update of Lassa fever outbreak in Nigeria for Week 45
SN 314 → 2020-W10: An update of Lassa fever outbreak in Nigeria for Week 44
SN 315 → 2020-W11: An update of Lassa fever outbreak in Nigeria for Week 43
SN 316 → 2020-W12: An update of Lassa fever outbreak in Nigeria for Week 42
SN 317 → 2020-W13: An update of Lassa fever outbreak in Nigeria for Week 41
SN 318 → 2020

In [8]:
import PyPDF2
import re

def extract_table1_data(pdf_path):
    """
    Extract Table 1 data from Lassa Fever PDF report.
    
    Args:
        pdf_path (str): Path to the PDF file
        
    Returns:
        dict: Dictionary containing all Table 1 fields
    """
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            first_page = pdf_reader.pages[0]
            text = first_page.extract_text()
            
            # Extract Table 1 data - pattern for the data row
            # Format: "98 (80) 18 2 11.1% State: 5 LGA: 11"
            table1_pattern = r'(\d+)\s+\((\d+)\)\s+(\d+)\s+(\d+)\s+([\d.]+)%\s+State:\s*(\d+)\s+LGA:\s*(\d+)'
            match = re.search(table1_pattern, text)
            
            if match:
                return {
                    'suspected_cases': int(match.group(1)),
                    'negative_cases': int(match.group(2)),
                    'confirmed_cases': int(match.group(3)),
                    'deaths': int(match.group(4)),
                    'case_fatality_rate': float(match.group(5)),
                    'states_affected': int(match.group(6)),
                    'lgas_affected': int(match.group(7))
                }
            else:
                return None
                
    except Exception as e:
        print(f"Error processing PDF: {str(e)}")
        return None


def extract_lassa_fever_data(pdf_path):
    """
    Extract Epi Week information and confirmed cases from Lassa Fever PDF report.
    
    Args:
        pdf_path (str): Path to the PDF file
        
    Returns:
        dict: Dictionary containing 'epi_week' and 'confirmed_cases'
    """
    try:
        # Open and read the PDF
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            
            # Extract text from first page (where the info is located)
            first_page = pdf_reader.pages[0]
            text = first_page.extract_text()
            
            # Extract Epi Week line - match the full date range
            epi_week_pattern = r'Epi Week \d+:[^\n]+'
            epi_week_match = re.search(epi_week_pattern, text)
            epi_week = epi_week_match.group(0).strip() if epi_week_match else None
            
            # Extract confirmed cases using the same logic as Table 1
            # Pattern: "98 (80) 18 2 11.1%" - confirmed cases is the 3rd number
            table1_pattern = r'(\d+)\s+\((\d+)\)\s+(\d+)\s+(\d+)\s+[\d.]+%'
            match = re.search(table1_pattern, text)
            confirmed_cases = int(match.group(3)) if match else None
            
            return {
                'epi_week': epi_week,
                'confirmed_cases': confirmed_cases
            }
            
    except FileNotFoundError:
        print(f"Error: File not found at {pdf_path}")
        return None
    except Exception as e:
        print(f"Error processing PDF: {str(e)}")
        return None


# Example usage
if __name__ == "__main__":
    # Replace with your PDF file path
    pdf_file_path = "/kaggle/working/ncdc_reports/2020-W02.pdf"
    
    # Extract basic data
    result = extract_lassa_fever_data(pdf_file_path)
    
    if result:
        print(f"Epi Week: {result['epi_week']}")
        print(f"Confirmed Cases: {result['confirmed_cases']}")
    else:
        print("Failed to extract data from PDF")
    
    print("\n" + "="*50)
    print("TABLE 1: Summary of current week indicators")
    print("="*50)
    
    # Extract and print Table 1
    table1_data = extract_table1_data(pdf_file_path)
    
    if table1_data:
        print(f"Suspected cases: {table1_data['suspected_cases']}")
        print(f"Negative cases: {table1_data['negative_cases']}")
        print(f"Confirmed cases: {table1_data['confirmed_cases']}")
        print(f"Deaths (Confirmed cases): {table1_data['deaths']}")
        print(f"Case Fatality Rate (CFR): {table1_data['case_fatality_rate']}%")
        print(f"States affected: {table1_data['states_affected']}")
        print(f"LGAs affected: {table1_data['lgas_affected']}")
    else:
        print("Failed to extract Table 1 data")

Epi Week: Epi Week 02: 06 – 12 January 2020
Confirmed Cases: 64

TABLE 1: Summary of current week indicators
Suspected cases: 158
Negative cases: 94
Confirmed cases: 64
Deaths (Confirmed cases): 12
Case Fatality Rate (CFR): 18.8%
States affected: 7
LGAs affected: 12


In [9]:
import PyPDF2
import re


def extract_lassa_fever_data(pdf_path):
    """
    Extract Epi Week information and confirmed cases from Lassa Fever PDF report.
    
    Args:
        pdf_path (str): Path to the PDF file
        
    Returns:
        dict: Dictionary containing 'epi_week' and 'confirmed_cases'
    """
    try:
        # Open and read the PDF
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            
            # Extract text from first page (where the info is located)
            first_page = pdf_reader.pages[0]
            text = first_page.extract_text()
            
            # Extract Epi Week line - match the full date range
            epi_week_pattern = r'Epi Week \d+:[^\n]+'
            epi_week_match = re.search(epi_week_pattern, text)
            epi_week = epi_week_match.group(0).strip() if epi_week_match else None
            
            # Extract confirmed cases using the same logic as Table 1
            # Pattern: "98 (80) 18 2 11.1%" - confirmed cases is the 3rd number
            table1_pattern = r'(\d+)\s+\((\d+)\)\s+(\d+)\s+(\d+)\s+[\d.]+%'
            match = re.search(table1_pattern, text)
            confirmed_cases = int(match.group(3)) if match else None
            
            return {
                'epi_week': epi_week,
                'confirmed_cases': confirmed_cases
            }
            
    except FileNotFoundError:
        print(f"Error: File not found at {pdf_path}")
        return None
    except Exception as e:
        print(f"Error processing PDF: {str(e)}")
        return None


# Example usage
if __name__ == "__main__":
    # Replace with your PDF file path
    pdf_file_path = "/kaggle/working/ncdc_reports/2020-W02.pdf"
    
    # Extract basic data
    result = extract_lassa_fever_data(pdf_file_path)
    
    if result:
        print(f"Epi Week: {result['epi_week']}")
        print(f"Confirmed Cases: {result['confirmed_cases']}")
    else:
        print("Failed to extract data from PDF")


Epi Week: Epi Week 02: 06 – 12 January 2020
Confirmed Cases: 64
